In [1]:
from pystlouisfed import FRED
import os
import numpy as np
import pandas as pd
import yfinance as yf

In [2]:
key = os.getenv("FRED_API_KEY")
fred = FRED(api_key=key)

In [3]:
cpi = fred.series_observations("CPIAUCSL", observation_start="1968-01-01")["value"].astype(float)
cpi.index = cpi.index.to_period("M")
infl_m = cpi.pct_change()
infl_yoy = cpi.pct_change(12)


tbill = fred.series_observations("TB3MS", observation_start="1968-01-01")["value"].astype(float) / 100 / 12
tbill.index = tbill.index.to_period("M")

In [4]:
sp = yf.download("^SP500TR", start="1987-12-01", interval="1mo",
                 auto_adjust=False, progress=False)["Close"].squeeze()
sp.index = sp.index.to_period("M")
eq_ret = sp.pct_change()

In [5]:
gsci = yf.download("^SPGSCI", start="1969-01-01", interval="1mo",
                   auto_adjust=False, progress=False)["Close"].squeeze()
gsci.index = gsci.index.to_period("M")


In [6]:
com_ret = gsci.pct_change() + tbill

df = pd.concat({
    "cpi": cpi,
    "infl_m": infl_m,
    "infl_yoy": infl_yoy,
    "tbill": tbill,
    "equity": eq_ret,
    "commodity": com_ret,
}, axis=1).loc["1970-01":].dropna()

df

,cpi,infl_m,infl_yoy,tbill,equity,commodity
1988-02,116.200,0.001724,0.039356,0.004717,0.046646,-0.012327
1988-03,116.500,0.002582,0.038324,0.004750,-0.030911,0.049991
1988-04,117.200,0.006009,0.039929,0.004925,0.011066,0.027180
1988-05,117.500,0.002560,0.039823,0.005217,0.008635,0.022244
1988-06,118.000,0.004255,0.039648,0.005383,0.045885,-0.032372
...,...,...,...,...,...,...
2026-05,333.979,0.004729,0.041666,0.003000,0.052633,-0.093118
2026-06,332.568,-0.004225,0.034635,0.003050,-0.009523,-0.105103
2026-07,332.813,0.000737,0.033039,0.003108,-0.000635,0.110785
2026-08,334.131,0.003960,0.033530,0.003100,0.027233,0.047624


In [7]:

for a in ["equity", "commodity"]:
    df[a + "_real"] = (1 + df[a]) / (1 + df["infl_m"]) - 1


df["high_q"] = df["infl_yoy"] >= df["infl_yoy"].quantile(0.8)

regime = "high_q"  
df["state"] = df[regime].map({True: "high", False: "normal"})


cmp = df.groupby("state")[["equity_real", "commodity_real"]].mean() * 12
cmp["n_months"] = df["state"].value_counts()
cmp

,equity_real,commodity_real,n_months
state,,,
high,0.029954,0.019080,93
normal,0.107454,0.072515,369


In [8]:
df["infl_yoy"].quantile(0.8)

np.float64(0.03734163385220213)

In [9]:
df

,cpi,infl_m,infl_yoy,tbill,equity,commodity,equity_real,commodity_real,high_q,state
1988-02,116.200,0.001724,0.039356,0.004717,0.046646,-0.012327,0.044845,-0.014027,True,high
1988-03,116.500,0.002582,0.038324,0.004750,-0.030911,0.049991,-0.033407,0.047287,True,high
1988-04,117.200,0.006009,0.039929,0.004925,0.011066,0.027180,0.005028,0.021045,True,high
1988-05,117.500,0.002560,0.039823,0.005217,0.008635,0.022244,0.006060,0.019634,True,high
1988-06,118.000,0.004255,0.039648,0.005383,0.045885,-0.032372,0.041453,-0.036472,True,high
...,...,...,...,...,...,...,...,...,...,...
2026-05,333.979,0.004729,0.041666,0.003000,0.052633,-0.093118,0.047678,-0.097387,True,high
2026-06,332.568,-0.004225,0.034635,0.003050,-0.009523,-0.105103,-0.005321,-0.101306,False,normal
2026-07,332.813,0.000737,0.033039,0.003108,-0.000635,0.110785,-0.001371,0.109967,False,normal
2026-08,334.131,0.003960,0.033530,0.003100,0.027233,0.047624,0.023181,0.043491,False,normal


In [10]:

import statsmodels.api as sm

rows = []
for a in ["equity", "commodity"]:
    for s, g in df.groupby("state"):
        X = sm.add_constant(g["infl_m"])
        ols = sm.OLS(g[a], X).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
        rows.append({
            "asset": a,
            "state": s,
            "n": len(g),
            "mean_real": g[a + "_real"].mean() * 12,
            "vol_real": g[a + "_real"].std() * np.sqrt(12),
            "beta": ols.params["infl_m"],
            "beta_se": ols.bse["infl_m"],
            "beta_t": ols.tvalues["infl_m"],
        })

res = pd.DataFrame(rows).set_index(["asset", "state"])
res.round(3)

n  mean_real  vol_real   beta  beta_se  beta_t
asset     state                                                   
equity    high     93      0.030     0.163 -1.898    2.067  -0.918
          normal  369      0.107     0.142  1.034    1.347   0.768
commodity high     93      0.019     0.210  4.388    2.149   2.042
          normal  369      0.073     0.206  8.818    1.401   6.295

In [11]:
yr = df.index.year

annual = pd.DataFrame({
    "inflation": (1 + df["infl_m"]).groupby(yr).prod() - 1,
    "equity": (1 + df["equity"]).groupby(yr).prod() - 1,
    "commodity": (1 + df["commodity"]).groupby(yr).prod() - 1,
    "tbill": (1 + df["tbill"]).groupby(yr).prod() - 1,
    "high_months": (df["state"] == "high").groupby(yr).sum()
})

annual["equity_real"] = (1 + annual["equity"]) / (1 + annual["inflation"]) - 1
annual["commodity_real"] = (1 + annual["commodity"]) / (1 + annual["inflation"]) - 1
annual["winner"] = np.where(annual["commodity_real"] > annual["equity_real"], "commodity", "equity")

annual.round(3)

,inflation,equity,commodity,tbill,high_months,equity_real,commodity_real,winner
1988,0.041,0.119,0.176,0.064,11,0.075,0.131,commodity
1989,0.046,0.317,0.218,0.084,12,0.258,0.164,equity
1990,0.063,-0.031,0.143,0.078,12,-0.088,0.076,commodity
1991,0.030,0.305,-0.151,0.055,8,0.267,-0.175,equity
1992,0.030,0.076,0.059,0.035,0,0.045,0.028,equity
1993,0.028,0.101,-0.069,0.030,0,0.071,-0.094,equity
1994,0.026,0.013,0.153,0.043,0,-0.012,0.124,commodity
1995,0.025,0.376,0.189,0.056,0,0.342,0.159,equity
1996,0.034,0.230,0.112,0.051,0,0.189,0.075,equity
1997,0.017,0.334,-0.141,0.052,0,0.311,-0.155,equity


In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PCT = {"infl_m", "infl_yoy", "tbill", "equity", "commodity", "equity_real", "commodity_real",
       "inflation", "mean_real", "vol_real"}
NUM = {"cpi", "beta", "beta_se", "beta_t"}

LABELS = {
    "date": "Date",
    "year": "Year",
    "asset": "Asset",
    "state": "Inflation regime",
    "cpi": "CPI",
    "infl_m": "Inflation (m/m)",
    "infl_yoy": "Inflation (y/y)",
    "inflation": "Inflation",
    "tbill": "T-bill return",
    "equity": "Equity return",
    "commodity": "Commodity return",
    "equity_real": "Equity real return",
    "commodity_real": "Commodity real return",
    "high_thr": "High inflation (y/y > 4%)",
    "high_q": "High inflation (top quintile)",
    "high_months": "High-inflation months",
    "n_months": "Months",
    "winner": "Higher real return",
    "n": "Months",
    "mean_real": "Mean real return (ann.)",
    "vol_real": "Real volatility (ann.)",
    "beta": "Inflation beta",
    "beta_se": "Beta std. error (HAC)",
    "beta_t": "Beta t-stat",
}

def _format_block(ws, cols, head_row, n_rows, widths):
    head_font = Font(name="Arial", bold=True, color="000000")
    head_fill = PatternFill("solid", start_color="F4B084")
    index_fill = PatternFill("solid", start_color="FCE4D6")
    body_font = Font(name="Arial", size=10)
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for j, name in enumerate(cols, start=1):
        label = LABELS.get(name, name)
        cell = ws.cell(row=head_row, column=j, value=label)
        cell.font, cell.fill, cell.border = head_font, head_fill, border
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        if name in PCT:
            fmt = "0.00%"
        elif name in NUM:
            fmt = "0.00"
        elif name == "date":
            fmt = "yyyy-mm"
        else:
            fmt = None
        for i in range(head_row + 1, head_row + 1 + n_rows):
            c = ws.cell(row=i, column=j)
            c.font, c.border = body_font, border
            if j == 1:
                c.fill = index_fill
            if fmt:
                c.number_format = fmt
        widths[j] = max(widths.get(j, 0), min(max(len(label), 10) + 2, 22))
    ws.row_dimensions[head_row].height = 30

def export_report(path, blocks, sheet="Report"):
    widths = {}
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        r = 1
        for title, d in blocks.items():
            d.to_excel(xw, sheet_name=sheet, index=False, startrow=r)
            ws = xw.sheets[sheet]
            t = ws.cell(row=r, column=1, value=title)
            t.font = Font(name="Arial", bold=True, size=14)
            _format_block(ws, list(d.columns), r + 1, len(d), widths)
            r += len(d) + 4
        for j, w in widths.items():
            ws.column_dimensions[get_column_letter(j)].width = w

d_out = df.copy()
d_out.index = d_out.index.to_timestamp()
d_out = d_out.rename_axis("date").reset_index()

export_report("inflation_hedge_report.xlsx", {
    "Metrics by regime": res.reset_index(),
    "Annual metrics": annual.rename_axis("year").reset_index(),
    "Monthly data": d_out,
})

: 